# AI Security Vulnerability Scanning Lab

This lab demonstrates how to analyze AI/ML code for security vulnerabilities using four tools:

- **Bandit** — Python static analysis
- **Semgrep** — pattern-based SAST
- **Pylint** — code quality and correctness
- **Safety** — dependency vulnerability scanning

All reports are saved into the `security-reports/` directory.

Your project files (e.g., `demo_code_1.py`, `demo_code_2.py`) must be in the same directory as this notebook.

In [ ]:
# Install required tools
!pip install bandit semgrep safety pylint -q

# Fix PATH for Codespaces/Colab so Semgrep is visible
import os
os.environ['PATH'] += f":{os.path.expanduser('~')}/.local/bin"

print('Environment ready. PATH updated.')

In [ ]:
# Ensure security-reports directory exists
import os
os.makedirs('security-reports', exist_ok=True)
os.listdir('.')

## Bandit — Static Analysis of Python Code

In [ ]:
# Run Bandit and save JSON output
!bandit -r . -f json -o security-reports/bandit.json
print('Bandit report saved to security-reports/bandit.json')

In [ ]:
# Load Bandit results
import json

with open('security-reports/bandit.json') as f:
    bandit = json.load(f)

len(bandit.get('results', []))

In [ ]:
# Display Bandit findings
bandit_findings = [
    {
        'severity': r.get('issue_severity'),
        'rule': r.get('test_id'),
        'cwe': r.get('issue_cwe', {}).get('id'),
        'file': r.get('filename'),
        'line': r.get('line_number'),
        'text': r.get('issue_text')
    }
    for r in bandit.get('results', [])
]
bandit_findings[:10]

In [ ]:
# Bandit severity counts
import pandas as pd
pd.Series([r['severity'] for r in bandit_findings]).value_counts()

## Semgrep — Pattern-Based SAST

In [ ]:
# Run Semgrep and save JSON output
!~/.local/bin/semgrep --config auto . --json --output security-reports/semgrep.json || semgrep --config auto . --json --output security-reports/semgrep.json
print('Semgrep report saved to security-reports/semgrep.json')

In [ ]:
# Load Semgrep results
with open('security-reports/semgrep.json') as f:
    semgrep = json.load(f)

len(semgrep.get('results', []))

In [ ]:
# Display Semgrep findings
semgrep_findings = [
    {
        'rule': r.get('check_id'),
        'file': r.get('path'),
        'line': r.get('start', {}).get('line'),
        'message': r.get('extra', {}).get('message')
    }
    for r in semgrep.get('results', [])
]
semgrep_findings[:10]

## Pylint — Code Quality and Correctness

In [ ]:
# Run Pylint and save output
!pylint *.py --disable=all --enable=W,E > security-reports/pylint.txt || true
print('Pylint report saved to security-reports/pylint.txt')

In [ ]:
# Show Pylint output
with open('security-reports/pylint.txt') as f:
    print(f.read())

## Safety — Dependency Vulnerability Scanning

In [ ]:
# Run Safety and save JSON output
!safety check --json > security-reports/safety.json || true
print('Safety report saved to security-reports/safety.json')

In [ ]:
# Load Safety results
with open('security-reports/safety.json') as f:
    safety = json.load(f)

safety.get('vulnerabilities', [])

## Consolidated Summary

In [ ]:
print('Bandit findings:', len(bandit_findings))
print('Semgrep findings:', len(semgrep_findings))
print('Safety vulnerabilities:', len(safety.get('vulnerabilities', [])))
print('Pylint report saved to security-reports/pylint.txt')